# Is Edmonton's Historical assessment dataset missing properties?

**Purpose:** let a human check, by hand, the claim in `data/DATA.md` §0 before it becomes a bug report to Edmonton Open Data.

**The claim.** Edmonton publishes two property-assessment datasets:

| dataset | id | covers |
|---|---|---|
| Property Assessment Data (Current Calendar Year) | `q7d6-ambg` | the live roll — assessment year **2025** |
| Property Assessment Data (Historical) | `qi6a-xuwt` | **2012–2025** |

They overlap: **both should describe assessment year 2025.** They don't agree. Whole buildings are in one and absent from the other.

Everything below hits the live API — no local data files, nothing from `data/raw/`. Run top to bottom.

> Figures in the markdown were measured **2026-07-28**. If a cell disagrees, the upstream data changed — that is itself worth knowing, and worth noting before filing.

In [ ]:
import json, urllib.parse, urllib.request
import pandas as pd

HIST = "qi6a-xuwt"   # Property Assessment Data (Historical)
CURR = "q7d6-ambg"   # Property Assessment Data (Current Calendar Year)

def soda(dataset, **params):
    """Query a Socrata dataset. Keys are passed as $-prefixed SoQL params.

    NOTE the default page size is 1,000 — always pass limit= explicitly or
    results silently truncate. That trap is why this helper exists.
    """
    url = f"https://data.edmonton.ca/resource/{dataset}.json?" + urllib.parse.urlencode(
        {("$" + k): v for k, v in params.items()}
    )
    with urllib.request.urlopen(url, timeout=180) as r:
        return json.load(r)

print("ready")

## 1. The two datasets disagree about Downtown, for the same year

The current roll is assessment year 2025 (see `data/DATA.md` §1 and the project's `status.json`). So this is a like-for-like comparison.

**Column-name gotcha:** the historical file calls it `neighbourhood_name`; the current roll calls it `neighbourhood`.

In [ ]:
# Current roll: pull Downtown rows and total them locally (sum() on this
# dataset's text-typed assessed_value is unreliable server-side).
cur = pd.DataFrame(soda(CURR, select="account_number,assessed_value,mill_class_1",
                        where="neighbourhood='DOWNTOWN'", limit=50000))
cur["assessed_value"] = pd.to_numeric(cur["assessed_value"], errors="coerce")

hist = pd.DataFrame(soda(HIST, select="account_number,assessed_value,mill_class_1",
                         where="neighbourhood_name='DOWNTOWN' AND assessment_year='2025'", limit=50000))
hist["assessed_value"] = pd.to_numeric(hist["assessed_value"], errors="coerce")

print(f"DOWNTOWN, assessment year 2025")
print(f"  current roll  (q7d6-ambg): {len(cur):>7,} accounts   ${cur.assessed_value.sum()/1e9:>5.2f}B")
print(f"  historical    (qi6a-xuwt): {len(hist):>7,} accounts   ${hist.assessed_value.sum()/1e9:>5.2f}B")
print(f"  MISSING from historical  : {len(cur)-len(hist):>7,} accounts   "
      f"${(cur.assessed_value.sum()-hist.assessed_value.sum())/1e9:>5.2f}B")

## 2. It's whole buildings, not scattered units

The two clearest cases are ICE District towers. Watch them appear every year, vanish in 2024, and stay vanished in 2025 — while sitting in the current roll the whole time.

- **10310 102 STREET NW** — Stantec Tower, Edmonton's tallest building
- **10360 102 STREET NW** — the adjacent ICE District residential tower

In [ ]:
for hn in ["10310", "10360"]:
    print(f"\n=== {hn} 102 STREET NW ===")
    h = soda(HIST, select="assessment_year,count(1) as n,sum(assessed_value) as v",
             where=f"house_number='{hn}' AND street_name='102 STREET NW'",
             group="assessment_year", order="assessment_year", limit=5000)
    print("  historical dataset, by year:")
    for r in h:
        print(f"    {r['assessment_year']}  {int(r['n']):>5,} accounts  ${float(r['v'] or 0)/1e6:>8,.1f}M")
    years = {r["assessment_year"] for r in h}
    for missing in ("2024", "2025"):
        if missing not in years:
            print(f"    {missing}  >>> NO ROWS AT ALL <<<")

    c = pd.DataFrame(soda(CURR, select="account_number,assessed_value,neighbourhood",
                          where=f"house_number='{hn}' AND street_name='102 STREET NW'", limit=5000))
    c["assessed_value"] = pd.to_numeric(c["assessed_value"], errors="coerce")
    print(f"  current roll (2025): {len(c):,} accounts  ${c.assessed_value.sum()/1e6:,.1f}M  "
          f"{set(c.neighbourhood)}")

## 3. Where did the 2023 accounts go? Nowhere.

The strongest form of the check, and the one that rules out the innocent explanations. Take every account in Downtown in **2023**, find the ones absent from Downtown in **2024**, then look for those account numbers **anywhere in Edmonton** in 2024.

If they'd been recoded to another neighbourhood, or a boundary moved, they would turn up elsewhere. They don't.

In [ ]:
def downtown(year):
    return pd.DataFrame(soda(HIST,
        select="account_number,mill_class_1,assessed_value,house_number,street_name",
        where=f"neighbourhood_name='DOWNTOWN' AND assessment_year='{year}'", limit=50000))

d23, d24 = downtown("2023"), downtown("2024")
gone = sorted(set(d23.account_number) - set(d24.account_number))
print(f"Downtown 2023: {len(d23):,}   2024: {len(d24):,}")
print(f"present 2023, absent 2024: {len(gone):,} accounts")

# Look for those exact account numbers ANYWHERE in Edmonton in 2024, batched.
found = {}
for i in range(0, len(gone), 200):
    inlist = ",".join("'" + a + "'" for a in gone[i:i+200])
    for r in soda(HIST, select="account_number,neighbourhood_name",
                  where=f"assessment_year='2024' AND account_number in({inlist})", limit=50000):
        found[r["account_number"]] = r.get("neighbourhood_name")

print(f"\n  still somewhere in the 2024 roll : {len(found):,}")
print(f"  absent from the 2024 roll ENTIRELY: {len(gone)-len(found):,}")
if found:
    print("  the survivors moved to:", pd.Series(list(found.values())).value_counts().to_dict())

In [ ]:
# What kind of properties vanished, and from which addresses?
g = d23[d23.account_number.isin(gone)].copy()
g["assessed_value"] = pd.to_numeric(g["assessed_value"], errors="coerce")
print("2023 value of the vanished accounts: "
      f"${g.assessed_value.sum()/1e9:.3f}B  ({len(g):,} accounts)\n")
print(g.mill_class_1.value_counts().to_string())
print("\ntop addresses among the vanished:")
print((g.house_number.fillna("?") + " " + g.street_name.fillna("?")).value_counts().head(8).to_string())

## 4. What it does to the story

Charting Downtown from the historical dataset alone shows a collapse that is about a third larger than the real one. Splice the current roll in for 2025 and the picture changes.

In [ ]:
s = pd.DataFrame(soda(HIST, select="assessment_year,count(1) as n,sum(assessed_value) as v",
                      where="neighbourhood_name='DOWNTOWN'",
                      group="assessment_year", order="assessment_year", limit=5000))
s["v"] = pd.to_numeric(s.v) / 1e9
s["n"] = pd.to_numeric(s.n)

peak = s.v.max()
hist_2025, curr_2025 = s.v.iloc[-1], cur.assessed_value.sum() / 1e9
print(s.to_string(index=False))
print(f"\npeak (historical)              : ${peak:.2f}B")
print(f"2025 per historical dataset    : ${hist_2025:.2f}B   -> {100*(hist_2025-peak)/peak:+.1f}% from peak")
print(f"2025 per CURRENT roll (correct): ${curr_2025:.2f}B   -> {100*(curr_2025-peak)/peak:+.1f}% from peak")

ax = s.set_index("assessment_year").v.plot(marker="o", figsize=(9, 4),
                                           title="Downtown total assessed value ($B)")
ax.scatter(["2025"], [curr_2025], color="red", zorder=5, s=70)
ax.annotate("2025 per the CURRENT roll", ("2025", curr_2025),
            textcoords="offset points", xytext=(-140, 6), color="red")
ax.set_xlabel("assessment year"); ax.grid(alpha=.3)

## 5. Scope: citywide, and now measured

**The earlier "~8,000 accounts citywide" figure was inferred from row counts and was misleading** — most of that gap is new construction, not defect. Measured account-by-account instead, the 7,929 net gap decomposes into:

| | accounts | what it is |
|---|---|---|
| in current roll, absent from historical 2025, **and not in 2023 either** | 8,171 | almost certainly new titles/construction — benign vintage difference |
| in historical 2025 but not the current roll | ~2,690 | demolitions/consolidations — benign |
| **in historical 2023 AND the current roll, ABSENT from historical 2025** | **2,448** | **the genuine defect** |

The 2,448 carry **$2.93B** and span **188 neighbourhoods** — citywide, though Downtown holds 1,292 (53%). Next worst: Magrath Heights 430, Glenora 269, Wîhkwêntôwin 124.

**They cluster at individual multi-unit addresses** — whole buildings vanish together, and not only downtown towers (7463 May Common NW in Magrath Heights, 14105 West Block Dr NW in Glenora). *State this as a symptom only; let the City diagnose the cause.*

The next cell re-derives the 2,448. It pulls ~1.3M account numbers in pages, so it takes a few minutes.


In [ ]:
# Re-derive the genuine defect count. Paged pulls -- a few minutes.
def pull(ds, where=None, label=""):
    out, off = set(), 0
    while True:
        p = {"select": "account_number", "order": "account_number",
             "limit": "50000", "offset": str(off)}
        if where: p["where"] = where
        rows = soda(ds, **p)
        if not rows: break
        out.update(r["account_number"] for r in rows if r.get("account_number"))
        off += 50000
        if len(rows) < 50000: break
    print(f"  {label}: {len(out):,}")
    return out

h23 = pull(HIST, "assessment_year='2023'", "historical 2023")
h25 = pull(HIST, "assessment_year='2025'", "historical 2025")
cur_all = pull(CURR, None, "current roll  ")

ghost  = (h23 & cur_all) - h25   # existed then, exists now, missing in between
newish = (cur_all - h25) - h23   # never in 2023 -- likely new construction
print(f"\nGENUINE DEFECT (in 2023 and now, absent from historical 2025): {len(ghost):,}")
print(f"likely-new accounts (explainable by snapshot vintage)          : {len(newish):,}")
